# Train the tray camera model

This notebook trains BuffetIQ's camera model on the group's tray photos. A YOLO26n-cls image classifier (the smallest current Ultralytics classification model) learns to sort a photo into Empty, Low, Half or Full. The photos are split into training (70%), validation (15%, used to pick the best training round) and test (15%, never seen in training), so the reported accuracy is honest.

In Colab, choose **Runtime → Change runtime type → T4 GPU** first. Training takes a few minutes. It also runs on a laptop CPU, more slowly.

## 1. Get the code

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/BuffetIQ'):
        !git clone -b claude/zen-keller-p86mcy https://github.com/stamedytweet-bit/BuffetIQ.git /content/BuffetIQ
    %cd /content/BuffetIQ
    !pip install -q -r requirements.txt
root = os.getcwd() if os.path.exists('config.yaml') else os.path.abspath('..')
os.chdir(root)
sys.path.insert(0, os.path.join(root, 'src'))
print(root)

## 2. Add the photos

On the laptop, zip the photo folder from the BuffetIQ folder with `zip -r tray_photos.zip tray_photos`. In Colab, run the next cell and choose `tray_photos.zip` when asked. (Locally, the photos are already in `tray_photos/`, so the cell does nothing.)

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        !unzip -o -q "{name}" -d /content/BuffetIQ

from buffetiq import camera
report = camera.check_photos()
print(report['counts'])
print('\n'.join(report['problems']) or 'No problems found.')

## 3. Train and test

This trains for up to 30 rounds (stopping early if it stops improving), then measures accuracy on the held-out test photos and saves `models/tray_cls.pt`, `docs/phase4_camera_results.json` and the confusion matrix chart.

In [ ]:
assert report['ready'], 'Each level needs at least 50 photos first.'
results = camera.train(epochs=30)
print(f"Test accuracy: {results['accuracy']:.1%} on {results['test_photos']} unseen photos")
print('Per level:', {k: f'{v:.0%}' for k, v in results['per_level_accuracy'].items()})
print(f"Sure (60% or more) on {results['share_sure']:.0%} of test photos; right on {results['accuracy_when_sure']:.0%} of those")

In [ ]:
from IPython.display import Image, display
display(Image(filename='docs/charts/phase4_confusion_matrix.png'))

## 4. Bring the model back to the laptop

In Colab, this downloads the model and the results. Put `tray_cls.pt` into `Documents/BuffetIQ/models/` and the other two files into `docs/` and `docs/charts/` on the laptop, then restart the app.

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import files
    for f in ('models/tray_cls.pt', 'docs/phase4_camera_results.json', 'docs/charts/phase4_confusion_matrix.png'):
        files.download(f)